# 02 Data Cleaning and Feature Engineering

## 1. Notebook Purpose

This notebook converts the raw pharma sales datasets into cleaned and analysis-ready files. The main cleaning tasks are datetime conversion, column-name standardization, time-feature creation, total-sales calculation, basic value validation, and exporting cleaned datasets.

## 2. Import Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

## 3. Load Raw Datasets

The raw files are loaded from `data/raw/`. Each file remains separate because its time grain serves a different analytical purpose.

In [3]:
RAW_DATA_DIR = Path("../data/raw")

daily_df = pd.read_csv(RAW_DATA_DIR / "salesdaily.csv")
hourly_df = pd.read_csv(RAW_DATA_DIR / "saleshourly.csv")
weekly_df = pd.read_csv(RAW_DATA_DIR / "salesweekly.csv")
monthly_df = pd.read_csv(RAW_DATA_DIR / "salesmonthly.csv")

datasets = {
    "daily": daily_df,
    "hourly": hourly_df,
    "weekly": weekly_df,
    "monthly": monthly_df,
}

pd.DataFrame(
    [{"dataset": name, "rows": len(df), "columns": len(df.columns)} for name, df in datasets.items()]
)

,dataset,rows,columns
0,daily,2106,13
1,hourly,50532,13
2,weekly,302,9
3,monthly,70,9


## 4. Standardize Column Names

Column names are stripped of extra spaces, converted to lowercase, and separated with underscores. For example, `Weekday Name` becomes `weekday_name`.

In [4]:
for df in datasets.values():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )

for name, df in datasets.items():
    print(f"{name}: {df.columns.tolist()}")

daily: ['datum', 'm01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06', 'year', 'month', 'hour', 'weekday_name']
hourly: ['datum', 'm01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06', 'year', 'month', 'hour', 'weekday_name']
weekly: ['datum', 'm01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06']
monthly: ['datum', 'm01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06']


## 5. Convert Date Column

`datum` is converted permanently to pandas datetime format. Invalid values are retained as `NaT` so they can be counted rather than silently removed.

In [5]:
date_validation = []

for name, df in datasets.items():
    df["datum"] = pd.to_datetime(df["datum"], errors="coerce")
    date_validation.append({
        "dataset": name,
        "datum_dtype": str(df["datum"].dtype),
        "invalid_dates": int(df["datum"].isna().sum()),
        "minimum_date": df["datum"].min(),
        "maximum_date": df["datum"].max(),
    })

date_validation = pd.DataFrame(date_validation)
date_validation

,dataset,datum_dtype,invalid_dates,minimum_date,maximum_date
0,daily,datetime64[us],0,2014-01-02 00:00:00,2019-10-08 00:00:00
1,hourly,datetime64[us],0,2014-01-02 08:00:00,2019-10-08 19:00:00
2,weekly,datetime64[us],0,2014-01-05 00:00:00,2019-10-13 00:00:00
3,monthly,datetime64[us],0,2014-01-31 00:00:00,2019-10-31 00:00:00


## 6. Define Medicine Category Columns

The same eight product-group columns should be available in every dataset. The validation below stops the notebook if a required category is missing.

In [6]:
medicine_category_columns = [
    "m01ab", "m01ae", "n02ba", "n02be",
    "n05b", "n05c", "r03", "r06",
]

missing_category_columns = {
    name: [column for column in medicine_category_columns if column not in df.columns]
    for name, df in datasets.items()
}

category_validation = pd.DataFrame([
    {
        "dataset": name,
        "all_categories_present": len(missing_columns) == 0,
        "missing_categories": ", ".join(missing_columns) if missing_columns else "None",
    }
    for name, missing_columns in missing_category_columns.items()
])

display(category_validation)

if any(missing_category_columns.values()):
    raise ValueError(f"Required medicine columns are missing: {missing_category_columns}")

,dataset,all_categories_present,missing_categories
0,daily,True,None
1,hourly,True,None
2,weekly,True,None
3,monthly,True,None


## 7. Create Time Features

Existing raw calendar fields are removed and recreated from `datum`; they are not trusted as cleaning inputs. All datasets receive year, month, month name, and quarter. Day and weekday fields are relevant to daily and hourly data, while hour is recreated only for hourly data.

In [7]:
raw_calendar_columns = [
    "year", "month", "month_name", "quarter",
    "day", "weekday", "weekday_name", "hour",
]

for name, df in datasets.items():
    columns_to_rebuild = [column for column in raw_calendar_columns if column in df.columns]
    df.drop(columns=columns_to_rebuild, inplace=True)

    df["year"] = df["datum"].dt.year.astype("Int64")
    df["month"] = df["datum"].dt.month.astype("Int64")
    df["month_name"] = df["datum"].dt.month_name()
    df["quarter"] = df["datum"].dt.quarter.astype("Int64")

for name in ["daily", "hourly"]:
    datasets[name]["day"] = datasets[name]["datum"].dt.day.astype("Int64")
    datasets[name]["weekday"] = datasets[name]["datum"].dt.dayofweek.astype("Int64")
    datasets[name]["weekday_name"] = datasets[name]["datum"].dt.day_name()

hourly_df["hour"] = hourly_df["datum"].dt.hour.astype("Int64")

for name, df in datasets.items():
    created_features = [column for column in raw_calendar_columns if column in df.columns]
    print(f"{name}: {created_features}")

daily: ['year', 'month', 'month_name', 'quarter', 'day', 'weekday', 'weekday_name']
hourly: ['year', 'month', 'month_name', 'quarter', 'day', 'weekday', 'weekday_name', 'hour']
weekly: ['year', 'month', 'month_name', 'quarter']
monthly: ['year', 'month', 'month_name', 'quarter']


## 8. Create Total Sales Column

`total_sales` is the row-level sum of all eight medicine-category sales fields.

In [8]:
for df in datasets.values():
    df["total_sales"] = df[medicine_category_columns].sum(axis=1, min_count=1)

pd.DataFrame([
    {
        "dataset": name,
        "minimum_total_sales": df["total_sales"].min(),
        "maximum_total_sales": df["total_sales"].max(),
    }
    for name, df in datasets.items()
])

,dataset,minimum_total_sales,maximum_total_sales
0,daily,0.000,198.950000
1,hourly,0.000,37.000000
2,weekly,153.507,790.837167
3,monthly,1.000,3146.906000


## 9. Validate Sales Values

Quality checks flag records for review without removing or replacing them. An unusually high `n02be` value is defined separately for each time grain using the upper IQR fence: Q3 + 1.5 x IQR. A monthly record is flagged when two or more medicine categories contain zero.

In [9]:
quality_flags = {}
validation_results = []

for name, df in datasets.items():
    negative_sales = df[medicine_category_columns].lt(0).any(axis=1)
    zero_total_sales = df["total_sales"].eq(0)

    n02be_q1, n02be_q3 = np.percentile(df["n02be"].dropna(), [25, 75])
    n02be_upper_fence = n02be_q3 + 1.5 * (n02be_q3 - n02be_q1)
    high_n02be = df["n02be"].gt(n02be_upper_fence)

    multiple_category_zeros = (
        df[medicine_category_columns].eq(0).sum(axis=1).ge(2)
        if name == "monthly"
        else pd.Series(False, index=df.index)
    )

    quality_flags[name] = {
        "negative_sales": negative_sales,
        "zero_total_sales": zero_total_sales,
        "high_n02be": high_n02be,
        "multiple_category_zeros": multiple_category_zeros,
    }

    validation_results.append({
        "dataset": name,
        "rows_with_negative_sales": int(negative_sales.sum()),
        "zero_total_sales_rows": int(zero_total_sales.sum()),
        "n02be_upper_fence": round(float(n02be_upper_fence), 2),
        "high_n02be_rows": int(high_n02be.sum()),
        "monthly_rows_with_multiple_zeros": int(multiple_category_zeros.sum()),
    })

validation_summary = pd.DataFrame(validation_results)
validation_summary

,dataset,rows_with_negative_sales,zero_total_sales_rows,n02be_upper_fence,high_n02be_rows,monthly_rows_with_multiple_zeros
0,daily,0,26,67.25,48,0
1,hourly,0,23944,4.69,4189,0
2,weekly,0,0,407.23,4,0
3,monthly,0,0,1681.67,1,1


In [10]:
for name, flags in quality_flags.items():
    if flags["negative_sales"].any():
        print(f"{name}: rows with negative category sales")
        display(datasets[name].loc[flags["negative_sales"], ["datum", *medicine_category_columns]])

    if flags["zero_total_sales"].any():
        print(f"{name}: rows with zero total sales")
        display(datasets[name].loc[flags["zero_total_sales"], ["datum", "total_sales"]])

    if flags["high_n02be"].any():
        print(f"{name}: sample of high n02be rows")
        display(datasets[name].loc[flags["high_n02be"], ["datum", "n02be"]].head(10))

monthly_zero_flags = quality_flags["monthly"]["multiple_category_zeros"]
print("Monthly records with two or more zero-value medicine categories")
display(monthly_df.loc[monthly_zero_flags, ["datum", *medicine_category_columns]])

daily: rows with zero total sales


,datum,total_sales
5,2014-01-07,0.0
108,2014-04-20,0.0
119,2014-05-01,0.0
351,2014-12-19,0.0
364,2015-01-01,0.0
370,2015-01-07,0.0
465,2015-04-12,0.0
716,2015-12-19,0.0
729,2016-01-01,0.0
735,2016-01-07,0.0


daily: sample of high n02be rows


,datum,n02be
41,2014-02-12,73.600
277,2014-10-06,69.875
360,2014-12-28,80.400
641,2015-10-05,70.950
647,2015-10-11,70.300
648,2015-10-12,68.850
653,2015-10-17,74.300
660,2015-10-24,67.400
668,2015-11-01,80.400
674,2015-11-07,69.900


hourly: rows with zero total sales


...

,datum,n02be
93,2015-10-18,423.174
144,2016-10-09,418.187
156,2017-01-01,546.899
264,2019-01-27,478.300


monthly: sample of high n02be rows


,datum,n02be
9,2014-10-31,1856.815


Monthly records with two or more zero-value medicine categories


,datum,m01ab,m01ae,n02ba,n02be,n05b,n05c,r03,r06
36,2017-01-31,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## 10. Final Cleaning Summary

This table provides a final structural and quality check before the cleaned files are exported.

In [11]:
cleaning_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "start_date": df["datum"].min(),
        "end_date": df["datum"].max(),
        "missing_values": int(df.isnull().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "zero_total_sales_rows": int(df["total_sales"].eq(0).sum()),
    }
    for name, df in datasets.items()
])

cleaning_summary

,dataset,rows,columns,start_date,end_date,missing_values,duplicate_rows,zero_total_sales_rows
0,daily,2106,17,2014-01-02 00:00:00,2019-10-08 00:00:00,0,0,26
1,hourly,50532,18,2014-01-02 08:00:00,2019-10-08 19:00:00,0,0,23944
2,weekly,302,14,2014-01-05 00:00:00,2019-10-13 00:00:00,0,0,0
3,monthly,70,14,2014-01-31 00:00:00,2019-10-31 00:00:00,0,0,0


## 11. Save Cleaned Files

The processed directory is created if needed. Each cleaned DataFrame is exported without its pandas index.

In [12]:
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_files = {
    "daily": "daily_cleaned.csv",
    "hourly": "hourly_cleaned.csv",
    "weekly": "weekly_cleaned.csv",
    "monthly": "monthly_cleaned.csv",
}

for name, file_name in output_files.items():
    output_path = PROCESSED_DATA_DIR / file_name
    datasets[name].to_csv(output_path, index=False)
    print(f"Saved {name}: {output_path}")

Saved daily: ..\data\processed\daily_cleaned.csv
Saved hourly: ..\data\processed\hourly_cleaned.csv
Saved weekly: ..\data\processed\weekly_cleaned.csv
Saved monthly: ..\data\processed\monthly_cleaned.csv
